# Adversarial Patch Experiments — YOLOv8 / YOLO11 / YOLO26

**Capstone: Person-Vanishing Adversarial Patch Attack against Ultralytics YOLO**

Trains patches directly against each model, then runs cross-version transfer tests.
Transfer results get distinct output directories so nothing gets overwritten.

**Before running:** `Runtime > Change runtime type > T4 GPU`

## 1. Setup

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
REPO_URL = 'https://github.com/Cmaddock99/Patch_Yolo.git'
REPO_DIR = '/content/Adversarial_Patch'
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install ultralytics opencv-python-headless tqdm -q

## 2. Direct Training — YOLOv8n

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolov8n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
from IPython.display import Image as IPImage, display
import json
display(IPImage('outputs/yolov8n_patch_v1/patches/patch.png', width=200))
with open('outputs/yolov8n_patch_v1/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 3. Direct Training — YOLO11n

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo11n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
import json
with open('outputs/yolo11n_patch_v1/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 4. Direct Training — YOLO26n

In [ ]:
import ultralytics
print('Ultralytics version:', ultralytics.__version__)
from ultralytics import YOLO
try:
    m = YOLO('yolo26n.pt')
    print('yolo26n loaded OK')
except Exception as e:
    print('Error:', e)

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo26n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
import json
with open('outputs/yolo26n_patch_v1/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 5. Cross-Version Transfer Tests

Patch trained on YOLOv8n — evaluated against YOLO11n and YOLO26n.
Results go to *separate* directories so training results are preserved.

Output dirs: `yolo11n_from_yolov8n_patch_v1_transfer/`, `yolo26n_from_yolov8n_patch_v1_transfer/`

In [ ]:
# v8 patch → v11
!python experiments/ultralytics_patch.py \
    --model yolo11n --eval-only \
    --load-patch outputs/yolov8n_patch_v1/patches/patch.png \
    --display 3

In [ ]:
# v8 patch → v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolov8n_patch_v1/patches/patch.png \
    --display 3

In [ ]:
# v11 patch → v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolo11n_patch_v1/patches/patch.png \
    --display 3

## 6. Results Summary

In [ ]:
import json, os
print(f"{'Experiment':<48} {'Clean':>6} {'Patched':>8} {'Suppression':>12}")
print('-' * 78)
for run_dir in sorted(os.listdir('outputs')):
    rp = f'outputs/{run_dir}/results.json'
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        tag = ' [TRANSFER]' if 'transfer' in run_dir else ' [DIRECT]  '
        label = run_dir + tag
        print(f"{label:<48} {r.get('clean_person_detections','-'):>6} "
              f"{r.get('patched_person_detections','-'):>8} "
              f"{str(r.get('detection_suppression_pct','-'))+'%':>12}")

## 7. Push Results to GitHub

In [ ]:
!git config user.email 'your@email.com'
!git config user.name 'Cmaddock99'
!git add outputs/ experiments/
!git commit -m 'Add experiment results: direct training + transfer tests v8/v11/v26'
TOKEN = 'YOUR_GITHUB_PAT_HERE'  # GitHub PAT with repo scope
REPO  = 'Cmaddock99/Patch_Yolo'
import subprocess
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{TOKEN}@github.com/{REPO}.git'])
!git push